# Face Comparison


## Setup


### Imports


In [1]:
from pathlib import Path

import numpy as np
import numpy.typing as npt


In [2]:
import keras
import tensorflow as tf

from keras import callbacks as k_callbacks
from keras import layers as k_layers
from keras import optimizers as k_optimisers
from keras import utils as k_utils
from tensorflow import config as tf_config # type: ignore
from tensorflow.data import AUTOTUNE as tf_AUTOTUNE # type: ignore
from tensorflow.data import Dataset as tf_Dataset # type: ignore


I0000 00:00:1779545370.893921  133264 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
from src.panopticon.recognition import ExampleModel
from src.panopticon.recognition._distance import euclidean_distance


Loaded Model 0
Loaded Model 1
Loaded Model 2
Loaded Model 3
Loaded Model 4
Loaded Model 5
Loaded Model 6
Loaded Model 7
Loaded Model 8
Loaded Model 9


### Settings


In [4]:
IMG_LENGTH = 64
IMG_SIZE = (IMG_LENGTH, IMG_LENGTH)
IMG_SHAPE = IMG_SIZE + (3,)

DS_PATH = Path('data/datasets/project_face_dataset')

# verification_data/00007133.jpg verification_data/00060449.jpg 1
# verification_data/00041961.jpg verification_data/00044353.jpg 0

PHOTO1 = DS_PATH / 'verification_data/00007133.jpg'
PHOTO2 = DS_PATH / 'verification_data/00060449.jpg'
PHOTO3 = DS_PATH / 'verification_data/00041961.jpg'
PHOTO4 = DS_PATH / 'verification_data/00044353.jpg'


### Config


In [5]:
GPUS: list[tf_config.PhysicalDevice] = tf_config.list_physical_devices(device_type='GPU')
print(f'{len(GPUS)} GPU(s): {GPUS}')

try:
	tf_config.experimental.set_memory_growth(GPUS[0], True)
except:
	# Invalid device or cannot modify virtual devices once initialized.
	pass


1 GPU(s): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


W0000 00:00:1779545372.378337  133264 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


## Loading


### Dataset


In [6]:
def load_images():
	ds = tf.data.Dataset.from_tensor_slices([
		str(PHOTO1),
		str(PHOTO2),
		str(PHOTO3),
		str(PHOTO4)
	])

	def load_image(filepath: tf.Tensor, /) -> tf.Tensor:
		raw_image: tf.Tensor = tf.io.read_file(filepath)
		# Can also use tf.image.decode_png
		image: tf.Tensor = tf.io.decode_png(raw_image, channels=3)
		image = tf.image.resize(image, IMG_SIZE)
		return image

	ds = ds.map(
		map_func=load_image,
		num_parallel_calls=tf_AUTOTUNE
	)

	ds = ds.batch(batch_size=4)

	def cast_image(image: tf.Tensor, /) -> tf.Tensor:
		return tf.cast(x=image, dtype=tf.float32)

	ds = ds.map(
		map_func=cast_image,
		num_parallel_calls=tf_AUTOTUNE
	)

	ds = ds.prefetch(buffer_size=tf_AUTOTUNE)

	return ds

dataset = load_images()


W0000 00:00:1779545372.406719  133264 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1779545372.697575  133264 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12877 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5080, pci bus id: 0000:07:00.0, compute capability: 12.0a


### Model


In [7]:
exmod = ExampleModel('Example')
exmod.load_model()
display(exmod.embedding_model.summary(show_trainable=True))


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ input_layer_1 (InputLayer)  │ (None, 64, 64, 3)     │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ efficientnetv2-b2           │ (None, 2, 2, 1408)    │  8,769,374 │   N   │
│ (Functional)                │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ global_average_pooling2d    │ (None, 1408)          │          0 │   -   │
│ (GlobalAveragePooling2D)    │                       │            │       │
└─────────────────────────────┴───────────────────────┴────────────┴───────┘

 Total params: 8,769,374 (33.45 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 8,769,374 (33.45 MB)

None

## Using


In [8]:
image_batch: tf.Tensor = dataset.as_numpy_iterator().next() # type: ignore
embedding_batch: tf.Tensor = exmod.embedding_model.predict_on_batch(image_batch)
display(embedding_batch.shape)


I0000 00:00:1779545376.216859  133329 service.cc:153] XLA service 0x7f2f1007c9a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779545376.216877  133329 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5080, Compute Capability 12.0a (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1779545376.323689  133329 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779545376.949439  133329 cuda_dnn.cc:461] Loaded cuDNN version 92200
E0000 00:00:1779545378.321084  133329 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1779545382.717292  133329 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


(4, 1408)

In [9]:
embeddings = np.array(embedding_batch)

distances: list[float] = []
for i in range(4):
	for j in range(4):
		if i == j:
			distances.append(0.0)
			continue
		distances.append(euclidean_distance(embeddings[i], embeddings[j]))

dist_matrix = np.array(distances).reshape(4, 4)
display(dist_matrix)


array([[ 0.        , 12.0171423 , 12.35990047, 12.18502712],
       [12.0171423 ,  0.        ,  9.42746449, 10.31695938],
       [12.35990047,  9.42746449,  0.        ,  8.51307869],
       [12.18502712, 10.31695938,  8.51307869,  0.        ]])